# Zero-shot baseline — Mozilla *Lost in Transcription* (Indonesian–Javanese)

**Purpose.** Establish the zero point before any fine-tuning happens, and answer
four questions that shape every later notebook. No training here. Accelerator:
**T4 x2** (single GPU used — see cell 1). Budget: this whole notebook should stay
under **3 GPU-hours**.

**Analogy.** This is giving a new hire a task *before* any training at all, just
to see how far their native ability goes. The result becomes the yardstick: if
after fine-tuning the model is not better than this, the training procedure is
the problem, not the model.

| # | question | why it matters |
|---|---|---|
| Q1 | Which model size is a viable starting point? | `whisper-small` vs having to go to `large-v3-turbo` |
| Q2 | Which language token — `id`, `jw`, or autodetect? | Whisper has a Javanese token (`jw`); the gap can be large and free |
| Q3 | How badly does Whisper normalize Javanese into Indonesian? | The feasibility-study failure mode, measured instead of assumed |
| Q4 | Seconds per clip on T4? | The submission container is 1x A100 80GB / 2h — this is the first latency number to compare against it |

**What notebook 02 already settled, and this notebook must not re-litigate**
(see `Hasil Notebook 02 - Data Build.md`, Gate 1 passed 25 Aug 2026):

- `text_official` is the only reference. `text_folded` is a measuring instrument only.
- Dev resolution is **±0.7-0.8 points at 95%**. A WER gap under ~1.5 points is not
  proven from a single run — use `scorer.compare_systems()` before trusting one.
- Two hard WER floors on dev: 11.8% of dev tokens never appear in train, and 0.86%
  is minority spelling variance. Dev will never approach zero.
- `jember-segments-16k` (owner `hamzapratama`) ships **208 FLAC, one per whole
  recording** — segments are addressed by sample offset, not by file. Dev split:
  20 recordings, 1,097 segments, 1.63 hours.

**Only `split == dev` is touched below. Train is never read in this notebook.**


## 0 — Dependencies

`transformers`, `torch`, `soundfile` ship with the Kaggle GPU image. `huggingface_hub`
is used only to read cached model sizes off disk for the latency table (Q4); if the
import fails, disk size is just reported as `None` and everything else still runs.


In [ ]:
!pip list 2>/dev/null | grep -Ei '^(transformers|torch|soundfile|huggingface-hub|pandas|numpy) ' || echo 'listing skipped'

## 1 — Setup, output convention, path discovery

Mirrors notebook 02's convention: content-based discovery (never a hardcoded depth
or slug — this account mounts owner-scoped, and `jember-segments-16k` belongs to
`hamzapratama`, not `thoriqfarras` like the three source corpora). `scorer.py` is
imported directly from the mounted `_meta/`, not rewritten here.


In [ ]:
# ============================================================
# CELL 1 - SETUP, OUTPUT CONVENTION, PATH DISCOVERY
# ============================================================
import os, sys, re, csv, json, time, random, shutil, collections
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

# ============================================================
# KNOBS
# ============================================================
# Full 5-model x 3-language grid is the brief's primary plan (15 predictions,
# each scored against 2 reference variants = 30 result rows). Whisper on T4 fp16
# runs well faster than real time even for large-v3-turbo (4-layer decoder), so
# the full grid over 1.63 h of dev audio should still land under the 3 GPU-hour
# budget. Flip to False only if a run turns out to be too slow: then only
# tiny/base/small get all 3 language tokens, and medium / large-v3-turbo get
# only the WINNING language token carried over from that subset (see the grid
# execution cell for how the winner is picked).
RUN_FULL_GRID = True

# Batch size used for the actual scoring predictions (not the Q4 latency probe,
# which times batch=1 and batch=8 separately regardless of this value).
BATCH_SIZE = 8

# How many dev clips the Q4 latency probe times, per model. Small on purpose -
# this measures steady-state seconds/clip, not the whole dev set.
LATENCY_N = 40

# SMOKE-RUN KNOB, NOT PART OF THE GRID. An int restricts DEV to the first N
# segments so the whole pipeline (both phases, winner selection, scoring, Q3,
# error tables, digest) can be proven to run end to end in a couple of minutes
# before committing GPU-hours to the real thing. Leave None for a real run.
SMOKE_N = None

SEED = 20260827
random.seed(SEED); np.random.seed(SEED)

FINDINGS = {}
# NOTE: this reset is why partial re-runs do not work. Re-running THIS cell
# discards everything the later cells collected. Always Run All.

def rec(section, **kw):
    FINDINGS.setdefault(section, {}).update(kw)

def head(title):
    print("\n" + "=" * 74); print(title); print("=" * 74)

def jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return round(float(o), 4)
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, float):          return round(o, 4)
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path): return str(o)
    return o

# ============================================================
# OUTPUT CONVENTION - same mirrored-tree rule as notebook 02.
# /kaggle/working/findings/<NOTEBOOK_ID>/  <->  <repo>/findings/<NOTEBOOK_ID>/
# ============================================================
NOTEBOOK_ID = "03-zeroshot"
WORKING     = Path("/kaggle/working")
RUN_TAG     = "-dryrun" if SMOKE_N else ""
OUT_DIR     = WORKING / "findings" / (NOTEBOOK_ID + RUN_TAG)
PRED_DIR    = OUT_DIR / "predictions"
ERR_DIR     = OUT_DIR / "error_tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
ERR_DIR.mkdir(parents=True, exist_ok=True)

def out(filename):
    p = OUT_DIR / filename
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

_ORIG_STDOUT = getattr(sys, "_nb_orig_stdout", sys.stdout)
sys._nb_orig_stdout = _ORIG_STDOUT

class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False

LOG_PATH = out("run-log.txt")
try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)

print("=" * 74)
print("RUN MODE   : %s      RUN_FULL_GRID = %s   SMOKE_N = %s"
      % ("SMOKE" if SMOKE_N else "REAL", RUN_FULL_GRID, SMOKE_N))
print("=" * 74)
if SMOKE_N:
    print("!! SMOKE RUN on the first %d dev segments only - numbers below mean" % SMOKE_N)
    print("!! nothing. Output goes to findings/%s%s/ so it can never be confused" % (NOTEBOOK_ID, RUN_TAG))
    print("!! with a real run.")
print("output dir : %s" % OUT_DIR)
print("console log: %s  (restarts every time this cell runs)" % LOG_PATH)
print("seed       : %d" % SEED)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device     : %s  (n_gpu=%d)" % (DEVICE, torch.cuda.device_count() if DEVICE == "cuda" else 0))
if DEVICE != "cuda":
    print("!!! No CUDA device visible. Select a GPU accelerator (T4 x2) in the")
    print("!!! notebook settings before a real run - medium/large-v3-turbo on CPU")
    print("!!! would take hours. A tiny SMOKE_N run is still fine on CPU.")
print("torch      :", torch.__version__)
import transformers
print("transformers:", transformers.__version__)

INPUT = Path("/kaggle/input")

# ------------------------------------------------------------------
# Discovery is CONTENT-BASED. jember-segments-16k mounts owner-scoped under
# hamzapratama (not thoriqfarras like the three source corpora), so the marker
# is recordings/_meta/manifest.csv sitting next to scorer.py and lexicon_v1.csv.
# ------------------------------------------------------------------
head("DISCOVERY")

manifest_hits = sorted(INPUT.glob("**/recordings/_meta/manifest.csv"))
if not manifest_hits:
    manifest_hits = [p for p in INPUT.glob("**/manifest.csv")
                      if (p.parent / "scorer.py").exists()
                      and (p.parent / "lexicon_v1.csv").exists()]
META_ROOT = manifest_hits[0].parent if manifest_hits else None
# META_ROOT = <root>/recordings/_meta ; row.path in manifest.csv already reads
# "recordings/<file>.flac", so the root to join against is two levels up.
AUDIO_ROOT = META_ROOT.parent.parent if META_ROOT else None

# ==================================================================
# MANUAL OVERRIDE - only if the line below says NOT FOUND.
# ==================================================================
# META_ROOT  = Path("/kaggle/input/datasets/hamzapratama/jember-segments-16k/recordings/_meta")
# AUDIO_ROOT = META_ROOT.parent.parent

print("META_ROOT  =", META_ROOT or "NOT FOUND")
print("AUDIO_ROOT =", AUDIO_ROOT or "NOT FOUND")
if META_ROOT is None:
    print("")
    print("!!! UNRESOLVED. Fill the MANUAL OVERRIDE block above and re-run this cell.")
else:
    print("")
    print("recordings/_meta/ :", sorted(p.name for p in META_ROOT.iterdir()))
    n_flac = len(list((AUDIO_ROOT / "recordings").glob("*.flac")))
    print("recordings/        : %d flac" % n_flac)

rec("env", notebook=NOTEBOOK_ID, seed=SEED, python=sys.version.split()[0],
    pandas=pd.__version__, numpy=np.__version__, torch=torch.__version__,
    transformers=transformers.__version__, device=DEVICE,
    n_gpu=(torch.cuda.device_count() if DEVICE == "cuda" else 0),
    meta_root=str(META_ROOT), audio_root=str(AUDIO_ROOT),
    run_full_grid=RUN_FULL_GRID, batch_size=BATCH_SIZE, latency_n=LATENCY_N,
    smoke_n=SMOKE_N)


## 2 — Import `scorer.py`, self-test

Imported straight from the mount, not rewritten — this notebook must use exactly
the module notebook 02 shipped and verified (28/28 self-tests, both normalizer
bugs fixed, Gate 1 passed 25 Aug 2026).


In [ ]:
# ============================================================
# CELL 2 - IMPORT scorer.py FROM THE MOUNT
# ============================================================
head("SCORER")
sys.path.insert(0, str(META_ROOT))
import scorer
print("scorer version:", scorer.VERSION)
ok, checks = scorer.self_test(verbose=False)
print("self_test: %d/%d passed" % (sum(1 for _, c, _ in checks if c), len(checks)))
if not ok:
    for name, cond, detail in checks:
        if not cond:
            print("  FAILED:", name, detail)
    raise RuntimeError("scorer.py self-test failed - do not trust any WER below until this is green.")

rec("scorer", version=scorer.VERSION, self_test_passed=ok, n_checks=len(checks))


## 3 — Manifest, split, lexicon; the dev set

`manifest.csv`'s own `split` column already carries `dev`/`train` (frozen by
notebook 02's grouped, stratified split) — `split.csv` is loaded too, only to
cross-check recording coverage. **Train is not read into memory at all below.**


In [ ]:
# ============================================================
# CELL 3 - MANIFEST / SPLIT / LEXICON, DEV SET
# ============================================================
head("LOAD")
MANIFEST = pd.read_csv(META_ROOT / "manifest.csv", dtype=str, keep_default_na=False)
SPLIT    = pd.read_csv(META_ROOT / "split.csv", dtype=str, keep_default_na=False)
LEX      = scorer.load_lexicon(META_ROOT / "lexicon_v1.csv")

for c in ("start_sample", "end_sample"):
    MANIFEST[c] = MANIFEST[c].astype(int)

DEV = (MANIFEST[MANIFEST["split"] == "dev"]
       .sort_values("seg_id").reset_index(drop=True))

dev_recs_manifest = set(DEV["audio_filename"])
dev_recs_split = set(SPLIT.loc[SPLIT["split"] == "dev", "audio_filename"])
assert dev_recs_manifest == dev_recs_split, \
    "manifest dev recordings and split.csv dev recordings disagree: %r" % (
        dev_recs_manifest ^ dev_recs_split)

print("dev segments   :", len(DEV))
print("dev recordings :", len(dev_recs_manifest))
print("dev hours       : %.3f" % (DEV["dur_s"].astype(float).sum() / 3600))
print("lexicon entries :", len(LEX))

if SMOKE_N:
    DEV = DEV.head(SMOKE_N).reset_index(drop=True)
    print("")
    print("!! SMOKE RUN: DEV truncated to the first %d segments." % SMOKE_N)

print(DEV[["seg_id", "audio_filename", "dur_s", "text_official"]].head(5).to_string(index=False))

rec("data", n_dev_segments=len(DEV), n_dev_recordings=len(dev_recs_manifest),
    dev_hours=round(float(DEV["dur_s"].astype(float).sum()) / 3600, 4),
    n_lexicon_entries=len(LEX))


## 4 — Model grid, decode settings, helpers

Decode settings are locked identically across every run so results are
comparable: greedy (`num_beams=1`, `temperature=0.0`), `task="transcribe"`,
no timestamps, `condition_on_prev_tokens=False` (stops one clip's hallucination
bleeding into the next — clips are scored independently per competition rule R8
anyway). Whisper's language code for Javanese is **`jw`**, not `jv` — a wrong
code silently falls back to the model's default guess.


In [ ]:
# ============================================================
# CELL 4 - MODEL GRID, DECODE SETTINGS, HELPERS
# ============================================================
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# (short name used in filenames, HF hub id). Order matters: index 0-2 are the
# "always run all languages" tier when RUN_FULL_GRID is False.
MODELS = [
    ("whisper-tiny",           "openai/whisper-tiny"),
    ("whisper-base",           "openai/whisper-base"),
    ("whisper-small",          "openai/whisper-small"),
    ("whisper-medium",         "openai/whisper-medium"),
    ("whisper-large-v3-turbo", "openai/whisper-large-v3-turbo"),
]

# Whisper's Javanese token is "jw", not the ISO 639-1 "jv". None = autodetect.
LANGS = ["id", "jw", None]
LANG_LABEL = {"id": "id", "jw": "jw", None: "auto"}

BASE_GEN_KWARGS = dict(
    task="transcribe",
    num_beams=1,
    do_sample=False,
    temperature=0.0,
    condition_on_prev_tokens=False,
    return_timestamps=False,
)

# DECODE HYGIENE - default OFF so the 27 Aug grid stays reproducible.
# The 27 Aug run showed greedy decoding falls into repetition loops on a few
# recordings: rec 191 emitted 678 tokens against 299 reference tokens
# ("dulu dulu dulu dulu ..."), and insertions alone were 0.1268 of the 0.8960
# WER - about 14 % of all error, none of it acoustic. "dulu" was the single
# most frequent hypothesis token in the whole dev set (451 occurrences).
# Flip this to True for a targeted re-run of the selected model only.
DECODE_HYGIENE = False
if DECODE_HYGIENE:
    BASE_GEN_KWARGS["no_repeat_ngram_size"] = 4

_RE_LANG_TOK = re.compile(r"<\|([a-z]{2,3})\|>")
_LANG_PROBE = {"n": 0}   # prints the real decoder prefix once, for diagnosis


def read_segment(row):
    import soundfile as sf
    y, sr = sf.read(str(AUDIO_ROOT / row.path), start=row.start_sample,
                     stop=row.end_sample, dtype="float32")
    if sr != 16000:
        raise ValueError("unexpected sample rate %d for %s" % (sr, row.seg_id))
    return y


def disk_size_mb(hf_id):
    # Actual bytes on disk for the cached snapshot, not an estimate from
    # parameter count - the latter would miss tokenizer/config overhead and
    # get the fp32-vs-fp16 story wrong.
    try:
        from huggingface_hub import scan_cache_dir
        for repo in scan_cache_dir().repos:
            if repo.repo_id == hf_id:
                return round(repo.size_on_disk / 1e6, 1)
    except Exception:
        pass
    return None


def load_whisper(hf_id):
    t0 = time.time()
    processor = WhisperProcessor.from_pretrained(hf_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        hf_id, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
    model = model.to(DEVICE)
    model.eval()
    return processor, model, time.time() - t0


def transcribe_arrays(processor, model, arrays, lang):
    # WhisperFeatureExtractor pads/trims every clip to a fixed 30 s log-mel
    # window regardless of input length, so no attention_mask juggling is
    # needed for batches this short (every dev clip is <= 29 s).
    inputs = processor(arrays, sampling_rate=16000, return_tensors="pt")
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    feats = inputs.input_features.to(DEVICE, dtype=dtype)
    kw = dict(BASE_GEN_KWARGS)
    if lang is not None:
        kw["language"] = lang
    with torch.no_grad():
        pred_ids = model.generate(feats, **kw)
    texts = [t.strip() for t in processor.batch_decode(pred_ids, skip_special_tokens=True)]
    detected = None
    if lang is None:
        # <|startoftranscript|> is index 0, the language token is index 1 when
        # it was not forced - decode it directly instead of re-running detection.
        # transformers 5.0.0 returns ONLY the generated content from generate():
        # the decoder prefix is gone, so there is no language token at any index.
        # The 27 Aug probe proved it - position 0..3 were ['Gaku','Gkan','Gmau',
        # 'Gan'], plain words. Ask the model directly instead.
        detected = ["?"] * pred_ids.shape[0]
        try:
            lang_ids = model.detect_language(input_features=feats)
            toks = processor.tokenizer.convert_ids_to_tokens(
                lang_ids.flatten().tolist())
            detected = [(m.group(1) if m else "?")
                        for m in (_RE_LANG_TOK.match(t) for t in toks)]
        except Exception as e:
            if _LANG_PROBE["n"] < 1:
                print("    [probe] detect_language unavailable: %r" % (e,))
        if _LANG_PROBE["n"] < 1:
            print("    [probe] detected langs, first batch:", detected[:8])
            _LANG_PROBE["n"] += 1
    return texts, detected


def has_repeated_ngram(tokens, n=3, min_repeats=3):
    if len(tokens) < n * min_repeats:
        return False
    grams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    counts = collections.Counter(grams)
    return bool(counts) and max(counts.values()) >= min_repeats


def hallucination_stats(refs, hyps):
    n = len(refs)
    n_long = n_repeat = 0
    for r, h in zip(refs, hyps):
        rt, ht = scorer.tokenize(r), scorer.tokenize(h)
        if rt and len(ht) > 3 * len(rt):
            n_long += 1
        if has_repeated_ngram(ht):
            n_repeat += 1
    return dict(pct_long=100.0 * n_long / n if n else 0.0,
                pct_repeat_ngram=100.0 * n_repeat / n if n else 0.0)


def jv_class_substitutions(refs, hyps, lex):
    # Q3: among substitutions whose REFERENCE token is lexicon-class 'jv', how
    # many were replaced by a class-'id' hypothesis token? Deletions of a jv
    # word are a different failure (dropped, not translated) and excluded from
    # the rate on purpose - they show up in wer_by_origin's del_rate instead.
    pairs = collections.Counter()
    n_jv_sub = n_jv_to_id = 0
    for r, h in zip(refs, hyps):
        rt, ht = scorer.tokenize(r), scorer.tokenize(h)
        for op, ri, hi in scorer.align_words(rt, ht):
            if op != "sub" or ri is None:
                continue
            ref_tok = rt[ri]
            if lex.get(scorer.lex_key(ref_tok), "unk") != "jv":
                continue
            n_jv_sub += 1
            hyp_tok = ht[hi]
            pairs[(ref_tok, hyp_tok)] += 1
            if lex.get(scorer.lex_key(hyp_tok), "unk") == "id":
                n_jv_to_id += 1
    rate = n_jv_to_id / n_jv_sub if n_jv_sub else 0.0
    return rate, n_jv_sub, n_jv_to_id, pairs


print("MODELS:", [m for m, _ in MODELS])
print("LANGS :", [LANG_LABEL[l] for l in LANGS])
print("decode:", BASE_GEN_KWARGS)


## 5 — Latency probe + prediction runs (Q4 + the grid itself)

One function per model: load once, time it at batch=1 and batch=8 on a small
fixed subset (Q4), then run every language token assigned to that model over
the **full** dev set and write raw predictions. Model is unloaded before moving
to the next one so peak VRAM never has two models resident at once.


In [ ]:
# ============================================================
# CELL 5 - process_model(): load, time, transcribe, save
# ============================================================
def measure_latency(processor, model, rows, batch_size):
    arrays = [read_segment(r) for r in rows.itertuples()]
    # Warmup: first call pays CUDA context / kernel autotune cost that has
    # nothing to do with the model itself.
    transcribe_arrays(processor, model, arrays[:min(batch_size, len(arrays))], "id")
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for i in range(0, len(arrays), batch_size):
        transcribe_arrays(processor, model, arrays[i:i + batch_size], "id")
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / len(arrays)


def process_model(short, hf_id, langs_to_run):
    head("MODEL %s (%s)  langs=%s" % (short, hf_id, [LANG_LABEL[l] for l in langs_to_run]))
    processor, model, load_s = load_whisper(hf_id)
    lat_rows = DEV.iloc[:min(LATENCY_N, len(DEV))]
    sec_bs1 = measure_latency(processor, model, lat_rows, 1)
    sec_bs8 = measure_latency(processor, model, lat_rows, min(8, len(lat_rows)))
    disk_mb = disk_size_mb(hf_id)
    print("  load %.1fs   disk %s MB   sec/clip  bs1=%.3f  bs8=%.3f  (n=%d)"
          % (load_s, disk_mb, sec_bs1, sec_bs8, len(lat_rows)))

    for lang in langs_to_run:
        label = LANG_LABEL[lang]
        t0 = time.time()
        all_texts, all_detected = [], []
        for i in range(0, len(DEV), BATCH_SIZE):
            chunk = DEV.iloc[i:i + BATCH_SIZE]
            arrays = [read_segment(r) for r in chunk.itertuples()]
            texts, detected = transcribe_arrays(processor, model, arrays, lang)
            all_texts.extend(texts)
            all_detected.extend(detected if detected is not None else [None] * len(texts))
        run_s = time.time() - t0
        P = pd.DataFrame(dict(seg_id=DEV["seg_id"].tolist(), hypothesis=all_texts))
        if lang is None:
            P["detected_lang"] = all_detected
        P.to_csv(PRED_DIR / ("%s__%s.csv" % (short, label)), index=False)
        print("  [%-4s] %d clips in %6.1fs  (%.3f s/clip)" % (label, len(DEV), run_s, run_s / max(len(DEV), 1)))

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return dict(load_s=round(load_s, 2), disk_mb=disk_mb,
                sec_per_clip_bs1=round(sec_bs1, 4), sec_per_clip_bs8=round(sec_bs8, 4),
                n_latency_clips=len(lat_rows))


def pick_winner_lang(models_subset):
    # Efficiency path only: average WER (text_official) per language token
    # across the models that already have all 3, then carry the winner alone
    # to the remaining (bigger, more expensive) models.
    scores = collections.defaultdict(list)
    for short, _ in models_subset:
        for lang in LANGS:
            label = LANG_LABEL[lang]
            pf = PRED_DIR / ("%s__%s.csv" % (short, label))
            if not pf.exists():
                continue
            P = pd.read_csv(pf, dtype=str, keep_default_na=False).set_index("seg_id")
            hyps = DEV["seg_id"].map(P["hypothesis"]).tolist()
            w = scorer.wer(DEV["text_official"].tolist(), hyps)["wer"]
            scores[label].append(w)
    avg = {k: sum(v) / len(v) for k, v in scores.items() if v}
    winner_label = min(avg, key=avg.get)
    print("phase-A average WER by language token:", {k: round(v, 4) for k, v in avg.items()})
    print("winner ->", winner_label)
    inv = {v: k for k, v in LANG_LABEL.items()}
    return inv[winner_label]


## 6 — Run the grid

`RUN_FULL_GRID = True` (the default) runs every model against every language
token in one pass. Set it `False` beforehand if a run proves too slow — the
first three models still get the full language grid, and the winning token
alone carries to `medium` / `large-v3-turbo`.


In [ ]:
# ============================================================
# CELL 6 - GRID EXECUTION
# ============================================================
head("GRID EXECUTION")
LATENCY = {}

if RUN_FULL_GRID:
    for short, hf_id in MODELS:
        LATENCY[short] = process_model(short, hf_id, LANGS)
    winner_lang = None  # not needed - every model already has all 3 tokens
else:
    for short, hf_id in MODELS[:3]:
        LATENCY[short] = process_model(short, hf_id, LANGS)
    winner_lang = pick_winner_lang(MODELS[:3])
    for short, hf_id in MODELS[3:]:
        LATENCY[short] = process_model(short, hf_id, [winner_lang])

print("")
print("prediction files:", sorted(p.name for p in PRED_DIR.glob("*.csv")))
rec("latency", **LATENCY)
rec("grid_mode", run_full_grid=RUN_FULL_GRID,
    winner_lang_carried=(LANG_LABEL[winner_lang] if not RUN_FULL_GRID else None))


## 7 — Score every prediction against both reference variants

`zeroshot_results.csv`: one row per (model, lang, ref_variant). `wer_by_origin`
already gives the jv/id/both/unk breakdown and the cosmetic-substitution
percentage for free; the hallucination and jv→id metrics are added per row.


In [ ]:
# ============================================================
# CELL 7 - SCORING GRID
# ============================================================
head("SCORING GRID")

REF_COLS = {"text_official": DEV["text_official"], "text_folded": DEV["text_folded"]}
pred_files = sorted(PRED_DIR.glob("*.csv"))
rows = []
for pf in pred_files:
    short, lang_label = pf.stem.split("__")
    P = pd.read_csv(pf, dtype=str, keep_default_na=False).set_index("seg_id")
    hyps = DEV["seg_id"].map(P["hypothesis"]).tolist()
    missing = int(pd.isna(pd.Series(hyps)).sum())
    if missing:
        print("  !! %s: %d dev seg_ids missing from predictions, scored as empty" % (pf.name, missing))
        hyps = [h if isinstance(h, str) else "" for h in hyps]
    for ref_variant, ref_series in REF_COLS.items():
        refs = ref_series.tolist()
        m = scorer.wer_by_origin(refs, hyps, LEX)
        jv_rate, n_jv_sub, n_jv_to_id, _ = jv_class_substitutions(refs, hyps, LEX)
        hall = hallucination_stats(refs, hyps)
        rows.append(dict(
            model=short, lang=lang_label, ref_variant=ref_variant,
            wer=round(m["wer_total"], 4), wer_jv=round(m["wer_jv"], 4),
            wer_id=round(m["wer_id"], 4), wer_both=round(m["wer_both"], 4),
            wer_unk=round(m["wer_unk"], 4),
            sub_rate=round(m["sub_rate"], 4), del_rate=round(m["del_rate"], 4),
            ins_rate=round(m["ins_rate"], 4), cosmetic_sub_pct=m["cosmetic_sub_pct"],
            jv_to_id_sub_pct=round(100 * jv_rate, 2), n_jv_sub=n_jv_sub, n_jv_to_id=n_jv_to_id,
            pct_long_hyp=round(hall["pct_long"], 2), pct_repeat_ngram=round(hall["pct_repeat_ngram"], 2),
            n_utts=m["n_utts"], n_ref_tokens=m["n_ref_tokens"],
        ))

RESULTS = pd.DataFrame(rows).sort_values(["ref_variant", "wer"]).reset_index(drop=True)
RESULTS.to_csv(out("zeroshot_results.csv"), index=False)
print(RESULTS.to_string(index=False))
rec("grid", n_configs=len(pred_files), n_result_rows=len(RESULTS))


## 8 — Q1: the model-size curve

Not just who wins, but **where the curve goes flat** — that model size is the
right fine-tune starting point.


In [ ]:
# ============================================================
# CELL 8 - Q1: MODEL SIZE CURVE
# ============================================================
head("Q1 - MODEL SIZE CURVE (best language token per model, text_official)")

off = RESULTS[RESULTS["ref_variant"] == "text_official"].copy()
order = {m: i for i, (m, _) in enumerate(MODELS)}
best_per_model = (off.loc[off.groupby("model")["wer"].idxmin()]
                   .assign(_order=lambda d: d["model"].map(order))
                   .sort_values("_order").drop(columns="_order")
                   .reset_index(drop=True))
print(best_per_model[["model", "lang", "wer", "wer_jv", "wer_id", "cosmetic_sub_pct"]].to_string(index=False))

deltas = best_per_model["wer"].diff()
flat_at = None
for i in range(1, len(deltas)):
    if deltas.iloc[i] > -0.01:  # improved by less than 1 WER point (or got worse)
        flat_at = best_per_model["model"].iloc[i]
        break
if flat_at:
    print("\ncurve flattens starting at: %s (< 1.0 point over the previous size)" % flat_at)
else:
    print("\ncurve has not flattened across the grid - every step up in size still helps by >= 1.0 point.")

rec("q1", table=best_per_model[["model", "lang", "wer", "wer_jv", "wer_id"]].to_dict("records"),
    flattens_at=flat_at)


## 9 — Q2: language token, for the best model

`id` / `jw` / autodetect on total WER and the jv/id breakdown. For autodetect,
also the distribution of the language it actually guessed per clip.


In [ ]:
# ============================================================
# CELL 9 - Q2: LANGUAGE TOKEN
# ============================================================
# WHICH MODEL CARRIES FORWARD IS AN ELBOW DECISION, NOT "lowest WER".
# The brief (Q1) asks where the curve flattens, not who wins: fine-tuning the
# champion is wasted money if a much cheaper model is statistically
# indistinguishable from it on this dev set.
#
# BUG FIXED 26 Aug (caught by the smoke run): this line used to read
#   best_model = best_per_model["model"].iloc[0]
# but best_per_model is sorted by MODEL SIZE, so .iloc[0] always returned
# whisper-tiny no matter how bad it was - the dry run "selected" whisper-tiny
# at WER 2.62 while large-v3-turbo sat at 0.75, and cell 10 then printed the
# false claim that tiny "has the lowest WER".
#
# The test is a PAIRED bootstrap over the same dev utterances (scorer.
# compare_systems, zero GPU): a candidate is "not provably worse" when the 95 %
# CI of (candidate - champion) still contains zero. Cheapest such model wins.
def _hyps_for(model, lang):
    P = pd.read_csv(PRED_DIR / ("%s__%s.csv" % (model, lang)), dtype=str,
                    keep_default_na=False).set_index("seg_id")
    return [h if isinstance(h, str) else "" for h in DEV["seg_id"].map(P["hypothesis"])]

head("ELBOW - CHEAPEST MODEL NOT PROVABLY WORSE THAN THE CHAMPION")
REFS_OFF = DEV["text_official"].tolist()
champion = best_per_model.loc[best_per_model["wer"].idxmin()]
champ_hyps = _hyps_for(champion["model"], champion["lang"])
_rows = []
for r in best_per_model.itertuples():
    if r.model == champion["model"]:
        _rows.append(dict(model=r.model, lang=r.lang, wer=round(r.wer, 4),
                          delta=0.0, delta_lo=0.0, delta_hi=0.0, not_worse=True))
        continue
    c = scorer.compare_systems(REFS_OFF, champ_hyps, _hyps_for(r.model, r.lang),
                               n_boot=1000, seed=SEED)
    _rows.append(dict(model=r.model, lang=r.lang, wer=round(r.wer, 4),
                      delta=round(c["delta"], 4), delta_lo=round(c["delta_lo"], 4),
                      delta_hi=round(c["delta_hi"], 4), not_worse=bool(c["delta_lo"] <= 0)))
ELBOW = pd.DataFrame(_rows)   # already in model-size order
print(ELBOW.to_string(index=False))
print("")
print("champion: %s / %s at WER %.4f" % (champion["model"], champion["lang"], champion["wer"]))
best_model = ELBOW.loc[ELBOW["not_worse"], "model"].iloc[0]
if best_model != champion["model"]:
    print("elbow   : %s is cheaper and its 95 %% CI against the champion still contains zero."
          % best_model)
else:
    print("elbow   : no cheaper model survives the test - the champion carries forward.")
rec("elbow", table=ELBOW.to_dict("records"), champion_model=str(champion["model"]),
    champion_lang=str(champion["lang"]), champion_wer=float(champion["wer"]),
    selected=str(best_model))

head("Q2 - LANGUAGE TOKEN (carried-forward model: %s)" % best_model)

lang_cmp = off[off["model"] == best_model][["lang", "wer", "wer_jv", "wer_id"]] \
    .set_index("lang").reindex(["id", "jw", "auto"])
print(lang_cmp.to_string())

# Same discipline as the elbow: a gap read off this table is not a result until
# it survives a paired test. On 27 Aug the raw table showed id 0.8960 vs jw
# 0.9184 and the notebook reported "id wins" - but the 95 % CI of that 2.2-point
# gap was [-0.129, +0.159], i.e. a coin flip. auto, by contrast, was real.
_ranked = off[off["model"] == best_model].sort_values("wer")
_top_lang = _ranked["lang"].iloc[0]
_top_hyps = _hyps_for(best_model, _top_lang)
_lang_tests = []
for _l in _ranked["lang"].iloc[1:]:
    c = scorer.compare_systems(REFS_OFF, _top_hyps, _hyps_for(best_model, _l),
                               n_boot=2000, seed=SEED)
    _lang_tests.append(dict(lang=_l, vs=_top_lang, delta=round(c["delta"], 4),
                            delta_lo=round(c["delta_lo"], 4),
                            delta_hi=round(c["delta_hi"], 4),
                            p_worse=round(1.0 - c["p_b_better"], 3),
                            separable=bool(c["delta_lo"] > 0)))
print("")
print("paired test against the best token (%s):" % _top_lang)
print(pd.DataFrame(_lang_tests).to_string(index=False))
_ties = [t["lang"] for t in _lang_tests if not t["separable"]]
if _ties:
    print("NOT separable from %s on this dev set: %s - report as a tie, not a win."
          % (_top_lang, ", ".join(_ties)))

auto_pred = PRED_DIR / ("%s__auto.csv" % best_model)
detected_dist = {}
if auto_pred.exists():
    P = pd.read_csv(auto_pred, dtype=str, keep_default_na=False)
    if "detected_lang" in P.columns:
        vc = P["detected_lang"].value_counts()
        detected_dist = vc.to_dict()
        print("\nautodetect - language token actually emitted, per clip:")
        print(vc.to_string())
        n_switch = int((P["detected_lang"] != vc.idxmax()).sum())
        print("clips NOT matching the modal detected language: %d / %d (%.1f%%)"
              % (n_switch, len(P), 100.0 * n_switch / max(len(P), 1)))

rec("q2", best_model=best_model, lang_comparison=lang_cmp.reset_index().to_dict("records"),
    detected_lang_distribution=detected_dist,
    best_lang=str(_top_lang), lang_paired_tests=_lang_tests,
    langs_tied_with_best=_ties)


## 10 — Decision: baseline model + language token for fine-tuning

Combines Q1 (size) and Q2 (language token) into the single choice notebooks 04
and 05 start from.


In [ ]:
# ============================================================
# CELL 10 - DECISION
# ============================================================
head("DECISION - BASELINE FOR FINE-TUNING")

best_lang_row = off[off["model"] == best_model].loc[off[off["model"] == best_model]["wer"].idxmin()]
DECISION = dict(model=best_model, lang=best_lang_row["lang"],
                wer=float(best_lang_row["wer"]), wer_jv=float(best_lang_row["wer_jv"]),
                wer_id=float(best_lang_row["wer_id"]),
                champion_model=str(champion["model"]), champion_wer=float(champion["wer"]),
                picked_champion=bool(best_model == champion["model"]))
print("selected model : %s" % DECISION["model"])
print("selected lang  : %s" % DECISION["lang"])
print("dev WER        : %.4f (jv %.4f, id %.4f)" % (DECISION["wer"], DECISION["wer_jv"], DECISION["wer_id"]))
print("")
if DECISION["model"] == champion["model"]:
    print("reasoning: %s has the lowest text_official WER in the grid, and no cheaper"
          % DECISION["model"])
    print("model survived the paired-bootstrap test against it.")
else:
    print("reasoning: the champion is %s at WER %.4f, but %s is cheaper and the 95 %% CI"
          % (champion["model"], champion["wer"], DECISION["model"]))
    print("of the difference still contains zero - see the elbow table above.")
if flat_at:
    print("(size curve flattens from %s onward.)" % flat_at)
print("language token '%s' beats the alternatives on this model - see the Q2 table above." % DECISION["lang"])

rec("decision", **DECISION)


## 11 — Q3: Javanese-to-Indonesian normalization, for the selected baseline

The measurement that becomes the **guard metric** reported in every notebook
after this one. If a later change lowers total WER but raises this number, it is
trading Javanese accuracy for Indonesian accuracy, not actually improving.


In [ ]:
# ============================================================
# CELL 11 - Q3: jv -> id SUBSTITUTIONS
# ============================================================
head("Q3 - JAVANESE TOKENS SUBSTITUTED WITH INDONESIAN (selected baseline)")

sel_pf = PRED_DIR / ("%s__%s.csv" % (DECISION["model"], DECISION["lang"]))
P = pd.read_csv(sel_pf, dtype=str, keep_default_na=False).set_index("seg_id")
sel_hyps = DEV["seg_id"].map(P["hypothesis"]).tolist()
sel_refs = DEV["text_official"].tolist()

jv_rate, n_jv_sub, n_jv_to_id, pairs = jv_class_substitutions(sel_refs, sel_hyps, LEX)
print("jv-class reference tokens substituted : %d" % n_jv_sub)
print("...of which the substitute is id-class: %d (%.2f%%)" % (n_jv_to_id, 100 * jv_rate))

sub_rows = []
for (ref_tok, hyp_tok), cnt in pairs.most_common(60):
    sub_rows.append(dict(ref=ref_tok, hyp=hyp_tok, count=cnt,
                          hyp_origin=LEX.get(scorer.lex_key(hyp_tok), "unk")))
SUB_DF = pd.DataFrame(sub_rows)
SUB_DF.to_csv(out("jv_to_id_substitutions.csv"), index=False)
print("\ntop substitutions of a jv-class reference token:")
print(SUB_DF.head(20).to_string(index=False))

rec("q3", jv_to_id_sub_pct=round(100 * jv_rate, 2), n_jv_sub=n_jv_sub, n_jv_to_id=n_jv_to_id)


## 12 — Q4: latency table


In [ ]:
# ============================================================
# CELL 12 - Q4: LATENCY
# ============================================================
head("Q4 - LATENCY (T4, fp16)")
lat_df = pd.DataFrame([dict(model=m, **v) for m, v in LATENCY.items()])
lat_order = {m: i for i, (m, _) in enumerate(MODELS)}
lat_df["_order"] = lat_df["model"].map(lat_order)
lat_df = lat_df.sort_values("_order").drop(columns="_order").reset_index(drop=True)
print(lat_df.to_string(index=False))
print("\nbatch=1 is the number relevant to the submission container (R8: clips scored independently).")

out("latency.json").write_text(json.dumps(jsonable(LATENCY), indent=1, sort_keys=True), encoding="utf-8")
print("saved ->", out("latency.json"))


## 13 — Cheap extras: raw vs folded, dev variance, bootstrap CI

All CPU, all free relative to the GPU spend above.


In [ ]:
# ============================================================
# CELL 13 - RAW VS FOLDED, DEV VARIANCE, BOOTSTRAP CI
# ============================================================
head("RAW (text_official) VS FOLDED (text_folded) - diacritic value at stake")
fold_cmp = RESULTS.pivot_table(index=["model", "lang"], columns="ref_variant", values="wer").reset_index()
fold_cmp["official_minus_folded"] = fold_cmp["text_official"] - fold_cmp["text_folded"]
print(fold_cmp.sort_values("official_minus_folded", ascending=False).to_string(index=False))

head("PER-RECORDING WER, selected baseline (%s / %s)" % (DECISION["model"], DECISION["lang"]))
tmp = DEV[["seg_id", "audio_filename", "text_official"]].copy()
tmp["hypothesis"] = sel_hyps
per_rec = []
for rec_id, g in tmp.groupby("audio_filename"):
    w = scorer.wer(g["text_official"].tolist(), g["hypothesis"].tolist())["wer"]
    per_rec.append(dict(audio_filename=rec_id, n_segs=len(g), wer=round(w, 4)))
PER_REC = pd.DataFrame(per_rec).sort_values("wer")
print(PER_REC.to_string(index=False))
print("\nstd across recordings: %.4f" % PER_REC["wer"].std())

head("BOOTSTRAP 95%% CI, selected baseline, text_official")
ci = scorer.wer_ci(sel_refs, sel_hyps, n_boot=1000, seed=SEED)
print("WER %.4f  [%.4f, %.4f]  half-width %.4f  (n_utts=%d, n_boot=%d)"
      % (ci["wer"], ci["lo"], ci["hi"], ci["half_width"], ci["n_utts"], ci["n_boot"]))

rec("dev_variance", per_recording=PER_REC.to_dict("records"), std_across_recordings=round(float(PER_REC["wer"].std()), 4))
rec("dev_ci", **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in ci.items()})


## 14 — Error tables for the 3 best configurations


In [ ]:
# ============================================================
# CELL 14 - ERROR TABLES, TOP 3 CONFIGS
# ============================================================
head("ERROR TABLES - TOP 3 CONFIGS (text_official)")
top3 = off.nsmallest(3, "wer")
for _, r in top3.iterrows():
    pf = PRED_DIR / ("%s__%s.csv" % (r["model"], r["lang"]))
    P = pd.read_csv(pf, dtype=str, keep_default_na=False).set_index("seg_id")
    hyps = DEV["seg_id"].map(P["hypothesis"]).tolist()
    et = scorer.error_table(DEV["text_official"].tolist(), hyps, LEX, top_k=50)
    fname = "%s__%s.csv" % (r["model"], r["lang"])
    if hasattr(et, "to_csv"):
        et.to_csv(ERR_DIR / fname, index=False)
    else:
        pd.DataFrame(et).to_csv(ERR_DIR / fname, index=False)
    print("\n%s (wer=%.4f):" % (fname, r["wer"]))
    print(et.head(10).to_string(index=False) if hasattr(et, "head") else et[:10])

rec("error_tables", configs=top3[["model", "lang", "wer"]].to_dict("records"))


## 15 — Self-validation

Each check below is a way this notebook could have failed silently, mapped
directly to the brief's completion criteria.


In [ ]:
# ============================================================
# CELL 15 - SELF-VALIDATION
# ============================================================
head("SELF-VALIDATION")
CHECKS = []
def check(name, cond, detail=""):
    CHECKS.append(dict(check=name, passed=bool(cond), detail=str(detail)[:160]))
    print("  [%s] %-58s %s" % ("PASS" if cond else "FAIL", name, detail))
    return bool(cond)

expected_configs = (len(MODELS) * len(LANGS)) if RUN_FULL_GRID else (3 * len(LANGS) + 2)
check("full WER table covers the intended grid, jv/id split present",
      len(pred_files) == expected_configs and {"wer_jv", "wer_id"} <= set(RESULTS.columns),
      "%d/%d configs" % (len(pred_files), expected_configs))
check("one (model, lang) selected as fine-tune baseline, with a reason",
      bool(DECISION.get("model")) and bool(DECISION.get("lang")))
# The check above only asserts a decision EXISTS. The 26 Aug smoke run passed
# it while selecting whisper-tiny at WER 2.62 over a 0.75 champion, because the
# selector read row 0 of a size-sorted table. This is the check that catches it.
_sel = ELBOW.loc[ELBOW["model"] == DECISION["model"]].iloc[0]
check("selected baseline is the champion, or provably not worse than it",
      bool(_sel["not_worse"]),
      "%s: delta %+.4f vs champion %s, 95%% CI [%+.4f, %+.4f]"
      % (DECISION["model"], _sel["delta"], DECISION["champion_model"],
         _sel["delta_lo"], _sel["delta_hi"]))
check("no cheaper model was skipped over",
      DECISION["model"] == ELBOW.loc[ELBOW["not_worse"], "model"].iloc[0],
      "cheapest surviving = %s" % ELBOW.loc[ELBOW["not_worse"], "model"].iloc[0])
_dist = FINDINGS.get("q2", {}).get("detected_lang_distribution") or {}
if (PRED_DIR / ("%s__auto.csv" % DECISION["model"])).exists():
    check("autodetect reported real language tokens, not '?'",
          bool(_dist) and "?" not in _dist, _dist)
else:
    print("  [skip] autodetect check - no auto prediction for the selected model")
check("jv_to_id substitution rate measured and saved",
      out("jv_to_id_substitutions.csv").exists() and n_jv_sub > 0,
      "%d jv-class substitutions" % n_jv_sub)
check("bootstrap 95% CI available for the selected baseline",
      ci["n_boot"] > 0 and ci["lo"] <= ci["wer"] <= ci["hi"],
      "half-width %.4f" % ci["half_width"])
check("latency.json filled for every model in the grid",
      set(LATENCY.keys()) == set(m for m, _ in MODELS),
      "%d/%d models" % (len(LATENCY), len(MODELS)))
check("not a smoke run", SMOKE_N is None,
      "SMOKE_N = %s%s" % (SMOKE_N, " - expected to FAIL on a smoke run" if SMOKE_N else ""))

n_pass = sum(1 for c in CHECKS if c["passed"])
print("\n%d/%d checks passed" % (n_pass, len(CHECKS)))
smoke_only = (SMOKE_N is not None and n_pass == len(CHECKS) - 1
              and not [c for c in CHECKS if not c["passed"] and c["check"] != "not a smoke run"])
if smoke_only:
    print(">>> SMOKE RUN OK. Pipeline runs end to end. Set SMOKE_N = None and Run All for real.")
elif n_pass == len(CHECKS):
    print(">>> ALL CHECKS PASSED. Baseline established - notebooks 04 and 05 may start.")
else:
    print(">>> BLOCKED - fix the FAIL rows above before treating any number here as final.")

rec("validation", checks=CHECKS, n_passed=n_pass, n_total=len(CHECKS), smoke_run=bool(SMOKE_N))


## 16 — Digest and packaging

Run last. Copy the marked block back into the chat. Outputs here are all small
CSV/JSON (no audio, no checkpoints), so — unlike notebook 02 — there is no
phase 2/3 split and no lite-zip: the whole `findings/03-zeroshot/` folder is
already the thing to download.


In [ ]:
# ============================================================
# CELL 16 - DIGEST + PACKAGING
# Run this last. Copy the ENTIRE block between the BEGIN/END markers and paste
# it back into the Claude conversation. That is the handoff.
# ============================================================
import hashlib
head("ARTIFACT FINGERPRINTS")
FP = {}
for p in sorted(OUT_DIR.rglob("*.csv")) + sorted(OUT_DIR.rglob("*.json")):
    if p.is_file():
        rel = p.relative_to(OUT_DIR)
        FP[str(rel)] = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
        print("  %-40s %s   %8.1f KB" % (rel, FP[str(rel)], p.stat().st_size / 1024))
rec("fingerprints", **FP)

EXPECTED = ["env", "scorer", "data", "latency", "grid_mode", "grid", "q1", "q2",
            "decision", "q3", "dev_variance", "dev_ci", "error_tables", "validation"]
missing_sections = [k for k in EXPECTED if k not in FINDINGS]
if missing_sections:
    print("")
    print("!" * 72)
    print("INCOMPLETE DIGEST - missing sections: %s" % missing_sections)
    print("Cause is almost always a partial re-run: cell 1 resets FINDINGS, so any")
    print("cell not executed after it contributes nothing. FIX: Run All.")
    print("!" * 72)

FINDINGS["_meta"] = dict(
    notebook="mozilla-javanese zero-shot baseline",
    notebook_id=NOTEBOOK_ID,
    generated_for="handoff to Claude - do not edit by hand",
    complete=(not missing_sections), missing_sections=missing_sections,
    artifacts=sorted(str(p.relative_to(OUT_DIR)) for p in OUT_DIR.rglob("*") if p.is_file()))

blob = json.dumps(jsonable(FINDINGS), ensure_ascii=False, separators=(",", ":"), sort_keys=True)
print("")
print("===== BEGIN CLAUDE-HANDOFF-JSON =====")
print(blob)
print("===== END CLAUDE-HANDOFF-JSON =====")
print("")
print("size: %.1f KB - select the whole block above and paste it into the chat" % (len(blob) / 1024))

try:
    out("dataset_findings.json").write_text(
        json.dumps(jsonable(FINDINGS), ensure_ascii=False, indent=1, sort_keys=True), encoding="utf-8")
    print("saved ->", out("dataset_findings.json"))
    sys.stdout.flush()
    print("saved ->", LOG_PATH)
except Exception as e:
    print("could not save:", e)

head("NEXT STEPS, BY HAND")
if SMOKE_N:
    print("THIS WAS A SMOKE RUN on %d dev segments. Every number above means nothing -" % SMOKE_N)
    print("this only proves the pipeline runs end to end. Output went to")
    print("findings/%s%s/, not the real folder. Do not download or commit any of it." % (NOTEBOOK_ID, RUN_TAG))
    print("")
    print("If the board in the self-validation cell showed no FAIL other than")
    print("'not a smoke run': set SMOKE_N = None in cell 1 and Run All for the real thing.")
else:
    print("1. Output tab -> Download all. Everything here is small CSV/JSON, so the")
    print("   whole archive should be a few MB.")
    print("")
    print("2. Unzip onto the repo root - it already contains findings/%s/." % NOTEBOOK_ID)
    print("")
    print("3. Paste the CLAUDE-HANDOFF-JSON block above into the chat.")
    print("")
    print("4. Write Hasil Notebook 03 - Zero-shot Baseline.md in the vault. Notebooks")
    print("   04 (Finetune Whisper Core) and 05 (XLS-R CTC + LM) can start in parallel")
    print("   once the (model, lang) decision above is reviewed.")
